## CHAT

In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)

from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 128  # Reducir batch si memoria es problema
    B = 150  # Era 300
    U = 300  # Era 600
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_96_192__",
        act_dim=3, 
        obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)

from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 128  # Reducir batch si memoria es problema
    B = 300
    U = 600
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_96_192__",
        act_dim=3, 
        obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)

from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 256
    
    U = 96
    B = 192
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_96_192__",
        act_dim=3, 
        obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
print(3)

In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)

from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 256
    
    U = 40
    B = 40
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_10_40",
        act_dim=3, 
        obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=10,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
# Celda para verificar los parámetros del scaler para las ACCIONES
import joblib
# Cargar el scaler (ajusta la ruta si es necesario)
scaler = joblib.load("../src/escalados/rl.lib")
# Ver TODOS los parámetros
print("=== PARÁMETROS COMPLETOS DEL SCALER ===")
print(f"data_min: {scaler.data_min_}")
print(f"data_max: {scaler.data_max_}")
print(f"scale: {scaler.scale_}")
print(f"min: {scaler.min_}")
# Los parámetros de ACCIONES son los índices 2, 3, 4
# (según TD3_Ray_test_macial.py líneas 270-271)
print("\n=== PARÁMETROS PARA ACCIONES (índices 2,3,4) ===")
print(f"Acción 0 (mf):     scale={scaler.scale_[2]:.4f}, min={scaler.min_[2]:.4f}")
print(f"Acción 1 (brk):    scale={scaler.scale_[3]:.4f}, min={scaler.min_[3]:.4f}")
print(f"Acción 2 (ice_sp): scale={scaler.scale_[4]:.4f}, min={scaler.min_[4]:.4f}")
# Verificar el rango FÍSICO de cada acción
print("\n=== RANGO FÍSICO DE ACCIONES ===")
for i, name in enumerate(["mf", "brk", "ice_sp"], start=2):
    # El MinMaxScaler normaliza: X_scaled = X * scale + min
    # Invirtiendo: X_physical = (X_01 - min) / scale
    # Pero con X_01 en [0,1], X_physical va de data_min a data_max
    print(f"{name}: [{scaler.data_min_[i]:.2f}, {scaler.data_max_[i]:.2f}]")
# VERIFICACIÓN CRÍTICA: ¿Qué valor físico da -1.0 y +1.0 escalado?
print("\n=== VERIFICACIÓN DE DESESCALADO ===")
action_scale = scaler.scale_[2:5]
action_min = scaler.min_[2:5]
for scaled_value in [-1.0, 0.0, 1.0]:
    action_01 = (scaled_value + 1.0) / 2.0  # De [-1,1] a [0,1]
    physical = action_01 * action_scale + action_min
    print(f"Scaled={scaled_value:+.1f} → mf={physical[0]:.1f}, brk={physical[1]:.1f}, ice_sp={physical[2]:.1f}")

In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)

from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 256
    
    U = 96
    B = 192
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_96_192___10",
        act_dim=3, 
        obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=10,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
print(3)

In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)

from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 128  # Reducir batch si memoria es problema
    B = 150  # Era 300
    U = 300  # Era 600
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_96_192__",
        act_dim=3, 
        obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
print(3)

In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)


from TD3_Ray_test_macial import *
from joblib import load
import ray  # <-- AÑADE ESTA LÍNEA
import time  # <-- También necesitas este para time.time()



from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 128  # Reducir batch si memoria es problema
    B = 150  # Era 300
    U = 300  # Era 600
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_96_192__",
        act_dim=3, 
#         obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)


from TD3_Ray_test_macial import *
from joblib import load
import ray  # <-- AÑADE ESTA LÍNEA
import time  # <-- También necesitas este para time.time()



from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 128  # Reducir batch si memoria es problema
    B = 150  # Era 300
    U = 300  # Era 600
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_96_192__",
        act_dim=3, 
#         obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)


from TD3_Ray_test_macial import *
from joblib import load
import ray  # <-- AÑADE ESTA LÍNEA
import time  # <-- También necesitas este para time.time()



from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 128  # Reducir batch si memoria es problema
    B = 150  # Era 300
    U = 300  # Era 600
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial_96_192__",
        act_dim=3, 
#         obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
        tau=0.001, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


In [ ]:
print(3)

In [ ]:
from ONNX_Predict.LSTM_onnx import LSTM_onnx 
from ONNX_Predict.Scaler_onnx import Scaler_onnx

from transition_function_model_macial import (
    setup_transition_function_model,
)


from TD3_Ray_test_macial import *
from joblib import load
import ray  # <-- AÑADE ESTA LÍNEA
import time  # <-- También necesitas este para time.time()



from TD3_Ray_test_macial import *
from joblib import load


import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but MinMaxScaler was fitted with feature names"
)



if __name__ == '__main__':
    #Definir las rutas a los modelos del entorno ---
    # Se han eliminado las comas al final para evitar errores.
    ICE_folder= "../macial/SHARE/CTTC_models/ONNX/ICE"
    PG_folder = "../macial/SHARE/CTTC_models/ONNX/PG"

#     # Crear la función de transición del entorno ---
#     # Se usan las variables correctas definidas arriba.
#     # Esta función es el "corazón" del entorno que el agente usará.
#     print("Configurando el entorno...")
#     t_function = setup_transition_function_model(ICE_folder, PG_folder)


    # Cargar el normalizador y preparar sus parámetros ---
    # Se carga el objeto scaler que contiene las estadísticas de normalización.
    print("Cargando los parámetros de escalado...")
    scaler = load("../src/escalados/rl.lib")

    # Se crea el diccionario de parámetros que las redes del agente necesitan.
    scaler_params = {
        "data_min": scaler.data_min_,
        "data_max": scaler.data_max_,
        "scale":    scaler.scale_,
        "min":      scaler.min_,
    }
    # Iniciar Ray (solo una vez por script)
    # Se configura para usar la memoria del sistema si es necesario (spilling)
    if ray.is_initialized():
        ray.shutdown()
    ray.init(object_store_memory=5 * 10**9) # Asigna 5 GB
    ray.init(log_to_driver=False, ignore_reinit_error=True)
    

    # --- Variables Base ---
    BATCH_SIZE = 128  # Reducir batch si memoria es problema
    B = 150  # Era 300
    U = 300  # Era 600
    
    print("\n--- INICIANDO ENTRENAMIENTO COMPLETO ---")
    training_start_time = time.time() # [NOU] Inicia el cronòmetre general
    
    # --- Creación e inicio del Learner ---
    td3_learner = TD3(
        model_paths=(ICE_folder, PG_folder), 
        version="discusion_macial__fi1",
        act_dim=3, 
#         obs_dim=5, 
        replay_size=1000000, 
        batch_size=BATCH_SIZE,
        gamma=0.99, 
#         tau=0.0005, 
        policy_noise=0.3, 
        noise_clip=0.5, 
        policy_delay=2, 
        scaler_params=scaler_params,
        vel_target=70,
        num_workers=3, # Usar los 3 núcleos de CPU
        U=U,
        B=B,
        early_stop=500,
        reuse_warmup_buffer= False
    )
    
    total_training_duration = time.time() - training_start_time # [NOU] Atura el cronòmetre

    # Iniciar el entrenamiento asíncrono
    
    td3_learner.learn(total_timesteps=100000, learning_starts=200, train_freq=U, gradient_steps=U)

    
    print("\n--- ENTRENAMIENTO TOTAL FINALIZADO ---")
    print(f"✅ El proceso de entrenamiento completo ha tardado: {total_training_duration:.2f} segundos ({total_training_duration / 60:.2f} minutos).") # [NOU] Imprimeix el resultat
    
    # Detener Ray al finalizar
    ray.shutdown()
    


2025-12-29 18:33:31.084820: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-29 18:33:31.103069: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-29 18:33:31.103085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-29 18:33:31.103771: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-29 18:33:31.107596: I tensorflow/core/platform/cpu_feature_guar

Cargando los parámetros de escalado...


2025-12-29 18:33:34,069	INFO worker.py:1927 -- Started a local Ray instance.
2025-12-29 18:33:34,837	INFO worker.py:1765 -- Calling ray.init() again after it has already been called.
2025-12-29 18:33:34.838325193 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136221, index: 10, mask: {11, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2025-12-29 18:33:34.843725644 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136220, index: 9, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2025-12-29 18:33:34.853474262 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136219, index: 8, mask: {9, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2025-12-29 18:33:34.85


--- INICIANDO ENTRENAMIENTO COMPLETO ---
Dispositivo de cómputo del LEARNER: cuda:0
Configurando el entorno...
⚙️ Configurando entorno ONNX...
   📍 ICE: ../macial/SHARE/CTTC_models/ONNX/ICE
   📍 PG:  ../macial/SHARE/CTTC_models/ONNX/PG


. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2025-12-29 18:33:35.709953736 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136236, index: 6, mask: {7, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2025-12-29 18:33:35.715669422 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136232, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2025-12-29 18:33:35.716773573 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136234, index: 4, mask: {5, }, error code: 22 error msg: Invalid argument. 

Configurando el correcto...
✅ 3 workers remotos creados.
🔥 Iniciando fase de calentamiento del buffer. Objetivo: 200 ventanas.
(RolloutWorker pid=4135629) Worker 0: Cargando modelos ONNX...
(RolloutWorker pid=4135629) ⚙️ Configurando entorno ONNX...
(RolloutWorker pid=4135629)    📍 ICE: ../macial/SHARE/CTTC_models/ONNX/ICE
(RolloutWorker pid=4135629)    📍 PG:  ../macial/SHARE/CTTC_models/ONNX/PG


(RolloutWorker pid=4135626) 2025-12-29 18:33:40.144320107 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136342, index: 7, mask: {8, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
(RolloutWorker pid=4135626) 2025-12-29 18:33:40.149324533 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136343, index: 8, mask: {9, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
(RolloutWorker pid=4135626) 2025-12-29 18:33:40.157336895 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinity_np failed for thread: 4136344, index: 9, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
(RolloutWorker pid=4135626) 2025-12-29 18:33:40.161353577 [E:onnxruntime:Default, env.cc:226 ThreadMain] pthread_setaffinit

(RolloutWorker pid=4135626) [Worker 1] Creado y listo.
(RolloutWorker pid=4135629) [Step 0115] Vel:  -0.2/70 km/h | Gas:  28.5  Freno:  27.7  RPM: 2406 | R: -1.420
(RolloutWorker pid=4135629) [Step 0347] Vel:  -0.2/70 km/h | Gas:  42.5  Freno:  53.9  RPM: 1689 | R: -2.719
Buffer: 6/200(RolloutWorker pid=4135629) [Step 0572] Vel:  -0.4/70 km/h | Gas:  29.3  Freno:  47.8  RPM: 1354 | R: -2.445
(RolloutWorker pid=4135629) [Step 0645] Vel:  -0.1/70 km/h | Gas:  39.4  Freno:  30.0  RPM: 2764 | R: -1.515
(RolloutWorker pid=4135629) [Step 0724] Vel:  -0.4/70 km/h | Gas:  44.6  Freno:  35.5  RPM: 2894 | R: -1.835
Buffer: 12/200(RolloutWorker pid=4135629) [Step 0014] Vel:   0.6/70 km/h | Gas:  31.1  Freno:  72.3  RPM: 1603 | R: -3.527
(RolloutWorker pid=4135625) Worker 2: Cargando modelos ONNX... [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.h

In [1]:
print(3)

3
